In [ ]:
from transformers import ViTForImageClassification, AutoImageProcessor, ImageClassificationPipeline
from pruna import smash, SmashConfig
import copy
import torch

In [ ]:
batch_size = 32
# batch_size = 2
runs = 50
# runs = 1
throw_out = 0.25
use_fp16 = False
verbose = True

model_id = "google/vit-large-patch16-224"
# model_id = "google/vit-base-patch16-224"

## Huggingface Models

In [ ]:
device = "cuda"
# Load a Vision Transformer
# timm_model = timm.create_model("vit_base_patch16_224", pretrained=True).to(device)
# model = torchvision.models.vit_b_16(pretrained=True)

hf_model = ViTForImageClassification.from_pretrained(model_id).to(device)
processor = AutoImageProcessor.from_pretrained(model_id)
hf_model.eval()


In [ ]:
r = 8

# Apply token merging
smash_config = SmashConfig(device=device)
smash_config["pruner"] = "token_merging"
smash_config["token_merging_r"] = r

smashed_model = smash(model=copy.deepcopy(hf_model), smash_config=smash_config)
smashed_model = smashed_model.to(device)
smashed_model.eval()

In [ ]:
from datasets import load_dataset

dataset = load_dataset("timm/mini-imagenet", split="test")

sample = dataset[2]
sample['image']

In [ ]:
input_tensor = processor(sample['image'], return_tensors="pt")['pixel_values'].to(device)
output = hf_model(input_tensor)

preds = output.logits.topk(5)
preds.indices

In [ ]:
output = smashed_model(input_tensor)
preds = output.logits.topk(5)
preds.indices

## HuggingFace Pipelines

In [ ]:
hf_pipeline = ImageClassificationPipeline(model=hf_model, image_processor=processor)
smashed_pipe = smash(model=copy.deepcopy(hf_pipeline), smash_config=smash_config)

In [ ]:
hf_pipeline(sample['image'])

In [ ]:
smashed_pipe(sample['image'])